#### Load Data into PostgreSQL

If you prefer to work with PostgreSQL, load the CSV into a database table and query it using `sql/src/db/queries`

Это — простой и мощный способ загрузить данные из файла в базу без написания SQL-запросов вручную

In [ ]:
# Настройка путей и импортов
import sys
from pathlib import Path

# Найдём корень проекта (где лежит README.md или config.py)
def find_project_root(marker="config.py"):
    current = Path().resolve()
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise RuntimeError(f"Project root with '{marker}' not found!")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

# Добавим корень проекта в sys.path, чтобы импортировать src и config
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Путь к данным
CSV_PATH = PROJECT_ROOT / "data" / "raw" / "third_wave_coffee_shop.csv"
assert CSV_PATH.exists(), f"CSV file not found at {CSV_PATH}"
print(f"CSV file: {CSV_PATH}")

#### Load CSV
**Run Query in Python:**

In [2]:
# Run query 

from src.db.queries import run_query

query = '''
SELECT time_of_day, coffee_name, sale_time
FROM third_wave_coffee_shop
LIMIT 3;
'''
df = run_query(query)
df


2025-11-17 19:45:51.227 | INFO     | src.db.queries:run_query:28 - Query returned 3 rows


,time_of_day,coffee_name,sale_time
0,Morning,Cappuccino,08:24:55
1,Morning,Cappuccino,08:24:55
2,Morning,Latte,08:42:32


In [4]:
query = '''
-- Общие метрики бизнеса
WITH tx AS (
    SELECT transaction_id, MAX(total_cost) AS total_cost
    FROM third_wave_coffee_shop 
    GROUP BY transaction_id
)
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(total_cost), 2) AS total_revenue,
    ROUND(AVG(total_cost), 2) AS avg_ticket
FROM tx;
'''
df = run_query(query)
df


2025-11-17 19:46:20.600 | INFO     | src.db.queries:run_query:28 - Query returned 1 rows


,total_orders,total_revenue,avg_ticket
0,3547,1668050.0,470.27
